# Traffic Demand Prediction
## AI-Powered Urban Traffic Congestion Analysis

**Objective:** Predict normalized traffic demand (0–1) for geographic locations using temporal, spatial, road, and weather features.

**Evaluation Metric:** `score = max(e, 100 * R²_score(actual, predicted))`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")


## 1. Load Dataset

In [ ]:
train = pd.read_csv('dataset/dataset/train.csv')
test  = pd.read_csv('dataset/dataset/test.csv')
sample_sub = pd.read_csv('dataset/dataset/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample submission shape:", sample_sub.shape)
train.head()


## 2. Exploratory Data Analysis

In [ ]:
print("=== Train Info ===")
print(train.dtypes)
print()
print("=== Missing Values (Train) ===")
print(train.isnull().sum())
print()
print("=== Demand Statistics ===")
print(train['demand'].describe())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Demand distribution
axes[0].hist(train['demand'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Demand Distribution')
axes[0].set_xlabel('Demand')
axes[0].set_ylabel('Frequency')

# Demand by Weather
train.groupby('Weather')['demand'].mean().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Average Demand by Weather')
axes[1].set_xlabel('Weather')
axes[1].set_ylabel('Mean Demand')
axes[1].tick_params(axis='x', rotation=45)

# Demand by RoadType
train.groupby('RoadType')['demand'].mean().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Average Demand by Road Type')
axes[2].set_xlabel('Road Type')
axes[2].set_ylabel('Mean Demand')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Demand by hour of day
train['hour'] = train['timestamp'].str.split(':').str[0].astype(int)
hourly = train.groupby('hour')['demand'].mean()

plt.figure(figsize=(12, 4))
hourly.plot(kind='line', marker='o', color='purple')
plt.title('Average Demand by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Mean Demand')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 3. Feature Engineering & Preprocessing

In [ ]:
def preprocess(df):
    df = df.copy()
    
    # --- Temporal Features ---
    ts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'] = ts[0]
    df['minute'] = ts[1]
    df['time_of_day'] = df['hour'] * 60 + df['minute']  # minutes since midnight
    
    # Cyclical encoding (captures circular nature: 23:00 close to 0:00)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['day_sin']  = np.sin(2 * np.pi * df['day'] / 7)
    df['day_cos']  = np.cos(2 * np.pi * df['day'] / 7)
    
    # --- Geospatial Features ---
    # Multi-resolution spatial encoding via geohash prefix lengths
    df['geo_prefix2'] = df['geohash'].str[:2]  # ~2500km x 2500km region
    df['geo_prefix3'] = df['geohash'].str[:3]  # ~156km x 156km region
    df['geo_prefix4'] = df['geohash'].str[:4]  # ~39km x 20km region
    
    # --- Encode Categoricals ---
    cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
                'geohash', 'geo_prefix2', 'geo_prefix3', 'geo_prefix4']
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype('category')
    
    # --- Fill Missing Numerical ---
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    
    return df

train = preprocess(train)
test  = preprocess(test)

print("Preprocessing complete!")
print("Train shape:", train.shape)
print("Test shape :", test.shape)


## 4. Model Training — LightGBM with 5-Fold Cross Validation

In [ ]:
features = [
    'day', 'hour', 'minute', 'time_of_day',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
    'NumberofLanes', 'Temperature',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
    'geohash', 'geo_prefix2', 'geo_prefix3', 'geo_prefix4'
]

X      = train[features]
y      = train['demand']
X_test = test[features]

print("Features used:", len(features))
print(features)


In [ ]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=128,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    model.fit(X_tr, y_tr)
    preds = np.clip(model.predict(X_val), 0, 1)
    score = r2_score(y_val, preds)
    cv_scores.append(score)
    print(f"Fold {fold} R² = {score:.4f}")

mean_r2 = np.mean(cv_scores)
comp_score = max(np.e, 100 * mean_r2)
print(f"\nMean CV R²         : {mean_r2:.4f} ± {np.std(cv_scores):.4f}")
print(f"Competition Score  : {comp_score:.4f}")


## 5. Feature Importance

In [ ]:
# Train final model on full training data
model.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 7))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
plt.title('LightGBM Feature Importances')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## 6. Generate Predictions & Save Submission

In [ ]:
test_preds = np.clip(model.predict(X_test), 0, 1)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission file saved!")
print("Shape:", submission.shape)
print()
print(submission.head(10))


In [ ]:
# Verify submission format matches sample_submission
sample_sub = pd.read_csv('dataset/dataset/sample_submission.csv')
print("Sample submission columns:", list(sample_sub.columns))
print("Our submission columns   :", list(submission.columns))
print()
print("Required size : 41778 x 2")
print("Our size      :", submission.shape[0], "x", submission.shape[1])
print()
print("Submission preview:")
submission.head()
